# Day 45 — Solutions

## Principal Component Analysis

**Boston University · Data Science Program**

---

**12 questions in four parts.**

| Part | | Questions |
|---|---|---|
| **A** | By hand | 1 – 3 |
| **B** | Scaling and components | 4 – 6 |
| **C** | Reading what came out | 7 – 9 |
| **D** | Judgement | 10 – 12 |

Run the setup cell first. Every question is self-contained after that.

> This is the solutions file. Each question shows working code, then the reasoning.
> Read the reasoning — most of the marks are there.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, adjusted_rand_score
from sklearn.datasets import load_wine, load_breast_cancer, load_digits, load_iris

TEAL, CORAL, VIOLET, BLUE, AMBER = "#1F9E86", "#E05C45", "#8268B5", "#3E7CB1", "#F0A530"
MUTED, GREY = "#7B8AA1", "#C9D2E0"
np.set_printoptions(precision=4, suppress=True)

# ---- Part A works on six points you can check with a ruler -------------
SIX = np.array([[2.0, 1.0], [3.0, 3.0], [4.0, 3.0],
                [5.0, 6.0], [6.0, 6.0], [7.0, 8.0]])

# ---- the rest uses real datasets ---------------------------------------
CX, CY = load_breast_cancer(return_X_y=True)
CN = load_breast_cancer().feature_names
print("six points:", SIX.shape, "   breast cancer:", CX.shape)

---
# Part A · By hand

## Question 1

Six points, small enough to check on paper:

```
(2,1)  (3,3)  (4,3)  (5,6)  (6,6)  (7,8)
```

**Centre them** (subtract the mean of each column), then build the covariance matrix
**yourself** with the formula

$$C_{jk} = \frac{1}{n-1}\sum_i (x_{ij} - \bar{x}_j)(x_{ik} - \bar{x}_k)$$

Print your matrix and check it against `np.cov`.

In [ ]:
mean = SIX.mean(axis=0)
X = SIX - mean
print("means:", mean)
print("centred:\n", X)

n = len(SIX)
C = (X.T @ X) / (n - 1)              # exactly the formula, written as matrices
print("\nmy C:\n", np.round(C, 4))
print("np.cov:\n", np.round(np.cov(SIX, rowvar=False), 4))
print("match:", np.allclose(C, np.cov(SIX, rowvar=False)))

# the same thing, element by element, to show it is not magic
C00 = sum((X[i, 0]) * (X[i, 0]) for i in range(n)) / (n - 1)
C01 = sum((X[i, 0]) * (X[i, 1]) for i in range(n)) / (n - 1)
print(f"\nby loop: C00 = {C00:.4f}   C01 = {C01:.4f}")

### Answer

```
means: [4.5 4.5]
my C:
 [[3.5 4.7]
  [4.7 6.7]]
match: True

by loop: C00 = 3.5000   C01 = 4.7000
```

**`(X.T @ X) / (n - 1)` is the formula**, just written with matrices instead of two loops.
Element `(j,k)` of `X.T @ X` is exactly $\sum_i x_{ij} x_{ik}$ over the centred data,
which is the sum in the definition.

Two things to notice:

- **Centring is not optional.** Covariance is defined on deviations from the mean. If you
  skip the subtraction you get $\sum x_{ij}x_{ik}$, which measures how far the data is
  from the *origin* — a quantity that changes if someone re-bases your units.
- **C is symmetric**, and it always will be: `C01` and `C10` are the same sum written in
  the other order. That symmetry is what guarantees the eigenvectors come out at right
  angles to each other.

## Question 2

Sweep the direction yourself. For angles 0° to 180°, project the centred points onto
`w = (cos θ, sin θ)` and measure the variance of the projection.

Print the best angle and its variance, plot the curve, and confirm that the variance at
your best angle equals `w.T @ C @ w`.

In [ ]:
angles = np.arange(0, 180, 0.25)
spreads = np.array([
    np.var(X @ np.array([np.cos(np.radians(a)), np.sin(np.radians(a))]), ddof=1)
    for a in angles
])

best = angles[spreads.argmax()]
print(f"best angle {best:.2f}°   variance {spreads.max():.4f}")
print(f"worst angle {angles[spreads.argmin()]:.2f}°   variance {spreads.min():.4f}")

wb = np.array([np.cos(np.radians(best)), np.sin(np.radians(best))])
print("\nw.T C w  =", round(float(wb @ C @ wb), 4))
print("total variance (trace of C) =", round(float(np.trace(C)), 4))

plt.figure(figsize=(6, 3))
plt.plot(angles, spreads, color=BLUE, lw=2.5)
plt.axvline(best, color=TEAL, ls="--")
plt.text(best + 4, spreads.max(), f" {best:.1f}° → {spreads.max():.2f}",
         color=TEAL, fontweight="bold", va="center")
plt.xticks([0, 45, 90, 135, 180]); plt.xlabel("direction"); plt.ylabel("variance")
plt.tight_layout(); plt.show()

### Answer

```
best angle 54.50°   variance 10.0648
worst angle 144.50°   variance 0.1352

w.T C w  = 10.0648
total variance (trace of C) = 10.2
```

**The best direction is 54.5°, not 45°.** It is worth seeing a value that is not a round
number — PCA returns whatever the data says, not a tidy diagonal. The second column has
the larger variance (6.7 against 3.5), so the winning direction tilts towards it.

**`w.T @ C @ w` reproduces the swept variance exactly**, to four decimals. That is the
point of the formula: the sweep is a *search*, the formula is an *evaluation*, and they
agree.

Check the two ends as well: **10.0648 + 0.1352 = 10.2000**, which is `C00 + C11` =
3.5 + 6.7. The best and worst directions between them account for every bit of variance
there is — none has gone missing, and none has been invented.

## Question 3

Now get the same answer from `np.linalg.eigh(C)` and from `sklearn`'s `PCA`.

Print all three directions and all three variances. Then explain, in your own words,
**what an eigenvector of the covariance matrix is**, using the curve you just plotted.

In [ ]:
evals, evecs = np.linalg.eigh(C)
order = evals.argsort()[::-1]
evals, evecs = evals[order], evecs[:, order]

ang = np.degrees(np.arctan2(evecs[1, 0], evecs[0, 0])) % 180
print(f"eigh   : {ang:.2f}°   variance {evals[0]:.4f}")
print(f"sweep  : {best:.2f}°   variance {spreads.max():.4f}")

p = PCA().fit(SIX)
ang_sk = np.degrees(np.arctan2(p.components_[0][1], p.components_[0][0])) % 180
print(f"sklearn: {ang_sk:.2f}°   variance {p.explained_variance_[0]:.4f}")

print("\nPC1 . PC2 =", round(float(evecs[:, 0] @ evecs[:, 1]), 12))
print("eigenvalues sum to", round(float(evals.sum()), 4),
      " vs trace(C) =", round(float(np.trace(C)), 4))
print("explained variance ratio:", p.explained_variance_ratio_.round(4))

### Answer

```
eigh   : 54.40°   variance 10.0649
sweep  : 54.50°   variance 10.0648
sklearn: 54.40°   variance 10.0649

PC1 . PC2 = -0.0
eigenvalues sum to 10.2  vs trace(C) = 10.2
explained variance ratio: [0.9868 0.0132]
```

**All three agree.** The 0.1° gap between the sweep and `eigh` is not a disagreement — it
is the resolution of the sweep. I stepped in 0.25° increments, so 54.50° is the closest
grid point to the true 54.40°, and the variance differs only in the fourth decimal. Make
the step smaller and the gap shrinks to nothing.

**What an eigenvector of C is, in the terms of that curve:**

> It is the direction at the **peak**. Nothing more mysterious than that.

The curve I plotted *is* the function $w \mapsto w^{T}Cw$ evaluated along every direction.
Its maximum is the top eigenvector and the height at the maximum is the eigenvalue. The
minimum is the second eigenvector. So "find the eigenvectors of C" and "find the best and
worst directions to project onto" are the same instruction, and `eigh` just does it
without searching.

Two facts the output confirms, both of which hold in any number of dimensions:

- **PC1 · PC2 = 0** exactly. The components are at right angles, so PC2 can only describe
  what PC1 did not. They never repeat each other.
- **The eigenvalues sum to the trace of C** (10.2 both ways), which is the total variance
  of the original columns. PCA rotates the variance into new axes; it does not create or
  destroy any. Here PC1 carries **98.7%** of it, so these six points are very nearly
  one-dimensional.

---
# Part B · Scaling and components

## Question 4

Break PCA with a unit change.

Take the breast cancer data and run PCA **without scaling**. Print PC1's explained
variance ratio and the three columns it loads on most heavily. Then do it **with**
`StandardScaler`, and compare.

In [ ]:
CZ = StandardScaler().fit_transform(CX)

for tag, D in [("WITHOUT scaling", CX), ("WITH scaling", CZ)]:
    p = PCA().fit(D)
    print(tag)
    print(f"   PC1 holds {p.explained_variance_ratio_[0]:.4%}")
    for i in np.abs(p.components_[0]).argsort()[::-1][:3]:
        print(f"     {CN[i]:26} loading {p.components_[0][i]:+.4f}")
    print()

sds = CX.std(axis=0, ddof=1)
print("largest column sd :", CN[sds.argmax()], round(sds.max(), 2))
print("smallest column sd:", CN[sds.argmin()], round(sds.min(), 5))
print("ratio:", int(round(sds.max() / sds.min())), "x")

### Answer

```
WITHOUT scaling
   PC1 holds 98.2045%
     worst area                 loading +0.8521
     mean area                  loading +0.5168
     area error                 loading +0.0557

WITH scaling
   PC1 holds 44.2720%
     mean concave points        loading +0.2609
     mean concavity             loading +0.2584
     worst concave points       loading +0.2509

largest column sd : worst area 569.36
smallest column sd: fractal dimension error 0.00265
ratio: 215171 x
```

**Unscaled, PC1 is 98.2% of the variance and it is essentially the two area columns.**
Look at the loadings: 0.852 and 0.517 on `worst area` and `mean area`, then a drop to
0.056. Twenty-seven other columns share what is left.

The reason is in the last three lines. The biggest column has a standard deviation of
**569**; the smallest has **0.00265** — a ratio of over **two hundred thousand**. Area is
measured in square pixels and fractal dimension is a ratio near zero. PCA chases variance,
so it found the column measured in the largest units and called it the answer.

**Scaled, PC1 drops to 44.3% and becomes something meaningful**: concave points and
concavity, which are the shape measurements a pathologist actually looks at.

Notice how badly the headline number misleads you. 98.2% *sounds* far better than 44.3%.
It is in fact a report that your preprocessing failed.

## Question 5

How many components does the breast cancer data deserve?

Draw the scree plot, print how many components reach 80%, 90% and 95%, and then check
those thresholds against what a model actually needs.

In [ ]:
p = PCA().fit(CZ)
cum = np.cumsum(p.explained_variance_ratio_)

for thr in [0.80, 0.90, 0.95]:
    print(f"{thr:.0%} of the variance: {np.argmax(cum >= thr) + 1} components")

plt.figure(figsize=(6.4, 3.2))
plt.bar(range(1, 31), p.explained_variance_ratio_ * 100, color=BLUE)
plt.plot(range(1, 31), cum * 100, color=CORAL, lw=2.2, marker="o", ms=3)
plt.axhline(95, color=TEAL, ls="--")
plt.xlabel("component"); plt.ylabel("% of variance")
plt.tight_layout(); plt.show()

print(f"\n{'k':>4}{'variance':>11}{'accuracy':>11}")
for k in [1, 2, 5, 10, 17, 30]:
    a = cross_val_score(LogisticRegression(max_iter=5000),
                        PCA(k).fit_transform(CZ), CY, cv=5).mean()
    print(f"{k:>4}{cum[k-1]:>11.1%}{a:>11.4f}")

a_raw = cross_val_score(LogisticRegression(max_iter=5000), CZ, CY, cv=5).mean()
print(f"\nall 30 original columns, no PCA: {a_raw:.4f}")

### Answer

```
80% of the variance: 5 components
90% of the variance: 7 components
95% of the variance: 10 components

   k   variance   accuracy
   1      44.3%     0.9122
   2      63.2%     0.9526
   5      84.7%     0.9719
  10      95.2%     0.9789
  17      99.1%     0.9772
  30     100.0%     0.9807

all 30 original columns, no PCA: 0.9807
```

**The thresholds give you a range: somewhere between 5 and 10 components.** The scree plot
has no sharp elbow — the bars decay smoothly — so reading an elbow off this chart is
guesswork, and three different people would read three different answers.

**The accuracy column does not rank the same way.** Read it carefully:

- It climbs steeply to k = 10 (**0.9789**), which is 95% of the variance.
- Then it **goes down** at k = 17 (0.9772) before recovering at 30 (0.9807).
- The best score, 0.9807, comes from keeping everything — the same as using the raw
  columns with no PCA at all.

That wobble between 10 and 30 is the important detail. It is not a trend; it is noise of
the order of one or two patients changing fold. **Differences that small are not evidence
of anything**, and if you picked k by taking the argmax of this column you would be fitting
the cross-validation rather than the data.

So the honest answer depends on what you want:

| If you want… | Keep |
|---|---|
| the best accuracy | all 30 — PCA buys you nothing here |
| a 2-D picture to look at | 2 (0.953, and you can plot it) |
| a good compromise | about 10 — 95% of the variance, 0.979, a third of the columns |

**The result worth sitting with: PCA did not help this model.** 0.9789 at ten components
against 0.9807 with everything is a fair trade if you want fewer columns, but it is a
*cost*, not a gain. 30 columns on 569 rows is not wide data, logistic regression handles it
comfortably, and there was no speed problem to solve. Dimensionality reduction is a tool
for a problem you actually have.

## Question 6

You have now seen a 95% threshold ask for 40 components on the digits (where 20 were
enough) and 10 here (where keeping all 30 still scored highest).

Explain **why a variance threshold cannot tell you how many components to keep**, and
describe what you would do instead. Be specific about what you would measure.

### Answer

**Why the threshold cannot answer the question.**

Explained variance measures **how much the data spreads**, in the units of the data. It is
computed from `X` alone. The number you actually care about is how well some *task* is
performed — and the task is nowhere in that computation. The threshold does not know
whether you are classifying tumours, drawing a picture, or clustering customers, so it
cannot know what counts as enough.

Concretely, the two are measuring different things:

- **Variance is about size of spread.** A component with large variance is one along which
  rows differ a lot.
- **Usefulness is about the task.** A component is useful if the differences it captures
  are the ones that matter for what you are doing.

Those usually overlap, which is why PCA works at all. But nothing forces them to, and the
two datasets above show the gap opening in both directions — on the digits the threshold
asked for **twice** what the model needed, while here it stopped at 10 when all 30 columns
still scored best.

There is also an arbitrariness problem that is easy to miss: **95% is a convention, not a
result.** Nobody derived it. Had the convention been 90% or 99% you would keep 7 or 17
components from the same data, with no more or less justification.

**What I would do instead.**

1. **Use the threshold to pick a shortlist, not a number.** The scree plot says "somewhere
   between about 5 and 15" — that is genuinely useful and takes one line.
2. **Then put a model on it and cross-validate across that shortlist**, exactly as the
   table above does. Plot accuracy against `k` and look for where the curve flattens.
3. **Measure the thing you actually care about.** If the goal is speed, measure prediction
   time as well and look for where the accuracy-per-millisecond is best. If the goal is a
   plot, the answer is 2 and no amount of variance accounting changes that.
4. **Fit the PCA inside the cross-validation**, in a `Pipeline`, not before it. Fitting PCA
   on all the data and then cross-validating leaks information from the test folds into
   the components. This is the same mistake we spent Day 36 on.

The one-line version: **a threshold narrows the search; the task decides the answer.**

---
# Part C · Reading what came out

## Question 7

Name the components.

Run PCA on the scaled breast cancer data and print PC1's loadings sorted by size. Then
look at the column names and say, in one sentence, what PC1 appears to be measuring.

In [ ]:
p = PCA().fit(CZ)
L = p.components_[0]
order = np.abs(L).argsort()[::-1]
print("largest loadings on PC1:")
for i in order[:10]:
    print(f"  {CN[i]:28} {L[i]:+.4f}")

print("\nsmallest loadings:")
for i in order[-4:]:
    print(f"  {CN[i]:28} {L[i]:+.4f}")

print("\nall loadings the same sign?", bool(np.all(L > 0) or np.all(L < 0)))
print("how many are positive:", int((L > 0).sum()), "of", len(L))

S = p.transform(CZ)
print(f"\nmean PC1 for malignant (0): {S[CY == 0, 0].mean():+.2f}")
print(f"mean PC1 for benign    (1): {S[CY == 1, 0].mean():+.2f}")

### Answer

```
largest loadings on PC1:
  mean concave points          +0.2609
  mean concavity               +0.2584
  worst concave points         +0.2509
  mean compactness             +0.2393
  worst perimeter              +0.2366
  worst concavity              +0.2288
  worst radius                 +0.2280
  mean perimeter               +0.2275
  worst area                   +0.2249
  mean area                    +0.2210

smallest loadings:
  mean fractal dimension       +0.0644
  symmetry error               +0.0425
  texture error                +0.0174
  smoothness error             +0.0145

all loadings the same sign? True
how many are positive: 30 of 30

mean PC1 for malignant (0): +3.71
mean PC1 for benign    (1): -2.21
```

**PC1 is measuring "how large and how irregular the tumour is" — a general severity score.**

Two pieces of evidence, and the second is the stronger one.

**First, the grouping.** The ten biggest loadings fall into two families that point the
same way:

- **size**: radius, perimeter, area — both the mean and the worst version of each
- **irregularity of the outline**: concave points, concavity, compactness

**Second, and this is the striking part: all 30 loadings are positive.** Not 26, not 28 —
every single one. PC1 is close to a plain weighted average of everything measured. That
only happens when *every* column in the dataset rises together, and it tells you something
real about the biology: a tumour that is large also tends to be lumpy, also tends to be
variable in texture. The columns are not 30 independent facts; they are 30 views of one
underlying thing, which is exactly the repetition PCA exists to collapse.

The four smallest loadings (0.0145 to 0.064) are the "error" columns — measures of
variability within a single image rather than of the tumour itself. PCA has effectively
decided they are not part of the main story, without being told.

**The last two lines confirm the reading, and they were not used to find it.** Malignant
cases average PC1 = **+3.71**, benign **−2.21**. PCA never saw the diagnosis; the direction
of greatest spread turned out to separate the classes nearly on its own, which is why PC1
alone scored 0.912 in Question 5.

One caution on naming: **the sign is arbitrary.** `sklearn` could equally have returned
−PC1, making every loading negative and malignant cases negative too. Only the *pattern*
of the loadings relative to each other carries meaning.

## Question 8

Rebuild the digits from a handful of numbers.

Load the digits, scale them, and reconstruct six images from 1, 3, 8 and 25 components.
Print the reconstruction error and the variance kept at each step. Show the images.

In [ ]:
DX, DY = load_digits(return_X_y=True)
scaler = StandardScaler().fit(DX)
DZ = scaler.transform(DX)
picks = [np.where(DY == d)[0][0] for d in [0, 2, 4, 6, 7, 9]]

fig, axes = plt.subplots(5, 6, figsize=(7, 6.4))
for j, idx in enumerate(picks):
    axes[0][j].imshow(DX[idx].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
axes[0][0].set_ylabel("original", fontsize=9)

for i, k in enumerate([1, 3, 8, 25], start=1):
    pk = PCA(k).fit(DZ)
    back = scaler.inverse_transform(pk.inverse_transform(pk.transform(DZ)))
    err = np.sqrt(((DX - back) ** 2).mean())
    print(f"{k:3d} components  keeps {pk.explained_variance_ratio_.sum():6.1%}"
          f"  error {err:.3f}")
    for j, idx in enumerate(picks):
        axes[i][j].imshow(back[idx].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    axes[i][0].set_ylabel(f"{k} comp", fontsize=9)

for row in axes:
    for ax in row:
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

print(f"\n64 numbers -> 8 numbers is a {64/8:.0f}x reduction")

### Answer

```
  1 components  keeps  12.0%  error 4.037
  3 components  keeps  30.0%  error 3.407
  8 components  keeps  52.9%  error 2.719
 25 components  keeps  85.1%  error 1.699
```

**One component gives you the same grey blob for every digit.** It has to — with k = 1
every image is `mean + (one number) × (one fixed picture)`, so the only thing that can
differ between two reconstructions is the brightness of that single picture.

**By 8 components the digits are recognisable**, and by 25 they are close to the originals
with just the fine strokes softened.

The useful idea here is what the reconstruction actually is. Each rebuilt image is

> the average digit, plus k fixed "component images", each scaled by one number

so **storing an image costs k numbers instead of 64** — and the component images are shared
by the whole dataset, paid for once.

Note the `scaler.inverse_transform`. `pca.inverse_transform` returns you to *standardised*
pixel space, not to pixels. Forget that second step and the images come back looking like
noise, because a standardised pixel of 0 means "average brightness", not "black".

## Question 9

Use PCA as a lens rather than a model.

The digits have 64 columns, so you cannot plot them. Reduce to 2 components, plot all
1,797 of them coloured by their true digit, and then say what the picture tells you that
a table of accuracy numbers would not.

In [ ]:
S2 = PCA(2).fit_transform(DZ)

plt.figure(figsize=(6.6, 5.6))
cmap = plt.cm.tab10
for d in range(10):
    m = DY == d
    plt.scatter(S2[m, 0], S2[m, 1], s=9, color=cmap(d), label=str(d))
plt.legend(ncol=5, fontsize=9, frameon=False, title="digit")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("1,797 handwritten digits, 64 columns squeezed into 2")
plt.tight_layout(); plt.show()

print("variance kept:", round(PCA(2).fit(DZ).explained_variance_ratio_.sum(), 4))

# which pairs actually sit on top of each other?
cent = np.array([S2[DY == d].mean(axis=0) for d in range(10)])
D = np.sqrt(((cent[:, None, :] - cent[None, :, :]) ** 2).sum(axis=2))
np.fill_diagonal(D, np.inf)
pairs = sorted({(min(i, j), max(i, j)): D[i, j] for i in range(10)
                for j in range(10) if i != j}.items(), key=lambda kv: kv[1])
print("\nclosest digit pairs in this 2-D picture:")
for (i, j), d in pairs[:4]:
    print(f"  {i} and {j}: centres {d:.2f} apart")

### Answer

```
variance kept: 0.2159

closest digit pairs in this 2-D picture:
  8 and 9: centres 0.16 apart
  5 and 8: centres 0.83 apart
  5 and 9: centres 0.87 apart
  1 and 8: centres 1.07 apart
```

**What the picture gives you that an accuracy table does not: it tells you *which* digits
are the problem, and *how* they fail.**

An accuracy of 0.544 from two components is a single number. It says the model is poor and
then stops talking. The plot says considerably more:

- **8 and 9 are sitting on top of each other.** Their centres are **0.16** apart — a
  fraction of the width of either cloud. In this view they are not two groups at all.
- **5, 8 and 9 form one tangle** in the middle of the picture (every one of the three
  closest pairs comes from that trio). That is where the errors will come from, and it is
  a specific, checkable claim.
- **0 and 6 sit out on their own**, far from the crowd. Those will almost never be confused
  with anything.
- **The clusters are elongated, not round**, and the spread *within* a digit is comparable
  to the distance *between* digits. That tells you immediately that a method assuming round,
  equally-sized clusters — K-Means, from Day 43 — is going to struggle here, and why.

All of that is actionable. "Check what is happening between 8, 9 and 5" is a decision you
can make from the picture and cannot make from the number 0.544.

**And note the honest caveat printed with it: this view holds only 21.6% of the variance.**
Digits that overlap here may well be separable in the other 62 directions — in fact we know
they are, because 20 components scored 0.970. So the plot is a tool for forming hypotheses
about your data, not for drawing conclusions about your model. Treat what you see as
"these look similar in the two biggest directions", not "these are inseparable".

---
# Part D · Judgement

## Question 10

Build a dataset where PCA throws away the answer.

Make three columns: two with large spread that say nothing about the label, and one with
small spread that determines it completely. Show that keeping only PC1 destroys the
model, and that keeping all three components does not.

In [ ]:
rg = np.random.default_rng(99)
n = 500
label = rg.integers(0, 2, n)
noise1 = rg.normal(0, 8.0, n)              # big spread, no information
noise2 = rg.normal(0, 6.0, n)              # big spread, no information
signal = label * 1.0 + rg.normal(0, 0.22, n)   # tiny spread, all the information
X3 = np.c_[noise1, noise2, signal]

p = PCA().fit(X3)
print("column sds:", X3.std(axis=0, ddof=1).round(3))
print("explained variance ratio:", p.explained_variance_ratio_.round(4))
print("PC1 loadings:", p.components_[0].round(4))
print("PC3 loadings:", p.components_[2].round(4))

S3 = p.transform(X3)
print()
for k in [1, 2, 3]:
    a = cross_val_score(LogisticRegression(max_iter=5000), S3[:, :k], label, cv=5).mean()
    print(f"keeping {k} component(s): accuracy {a:.4f}")

a3 = cross_val_score(LogisticRegression(max_iter=5000), S3[:, [2]], label, cv=5).mean()
print(f"\nkeeping ONLY PC3 (the smallest): accuracy {a3:.4f}")
print(f"keeping the raw columns        : "
      f"{cross_val_score(LogisticRegression(max_iter=5000), X3, label, cv=5).mean():.4f}")

### Answer

```
column sds: [8.266 5.991 0.531]
explained variance ratio: [0.6539 0.3435 0.0027]
PC1 loadings: [ 1.      0.006  -0.0046]
PC3 loadings: [ 0.0046 -0.005   1.    ]

keeping 1 component(s): accuracy 0.5420
keeping 2 component(s): accuracy 0.5520
keeping 3 component(s): accuracy 0.9880

keeping ONLY PC3 (the smallest): accuracy 0.9800
keeping the raw columns        : 0.9880
```

**PC1 and PC2 together hold 99.7% of the variance and score 0.552 — barely better than a
coin toss. PC3 holds 0.27% and scores 0.980 entirely on its own.**

The loadings say exactly what happened, and they are almost comically clean:

- **PC1 = `[1.000, 0.006, -0.005]`** — it is the first noise column, essentially untouched.
- **PC3 = `[0.005, -0.005, 1.000]`** — it is the signal column, essentially untouched.

PCA did not blend anything here. It simply **sorted the three columns by spread** and put
the only useful one last, where every rule of thumb tells you to delete it.

**A 95% variance rule keeps 2 components here and throws the model away** — from 0.988 down
to 0.552. So would any rule phrased in terms of variance, because the quantity those rules
measure has nothing to do with the quantity that matters.

How to build one of these on purpose, which is worth knowing because it tells you what to
watch for in real data:

1. Give the useless columns a large standard deviation (8.3 and 6.0).
2. Give the useful column a small one (0.53) — the two classes are centred at 0 and 1,
   while the noise within each class is only 0.22. **The classes are cleanly separated;
   they are just not *widely* separated.**

That second point is the whole trick. **Separation and spread are different things.** A
column can split two groups perfectly while varying over a tiny range — a binary flag, a
pass/fail threshold, a small but decisive measurement. Those are exactly the columns PCA
ranks last.

## Question 11

PCA before clustering: does it help?

Take the scaled digits and run `KMeans(10)` on the full 64 columns, and on 2, 10 and 20
components. Compare agreement with the true digits, using `adjusted_rand_score`.

In [ ]:
print(f"{'input':>18}{'agreement':>12}")
km = KMeans(10, n_init=10, random_state=0).fit(DZ)
print(f"{'all 64 columns':>18}{adjusted_rand_score(DY, km.labels_):>12.4f}")

for k in [2, 5, 10, 20, 30, 40]:
    Sk = PCA(k, random_state=0).fit_transform(DZ)
    lab = KMeans(10, n_init=10, random_state=0).fit(Sk).labels_
    print(f"{str(k) + ' components':>18}{adjusted_rand_score(DY, lab):>12.4f}")

### Answer

```
             input   agreement
    all 64 columns      0.4679
      2 components      0.3262
      5 components      0.4501
     10 components      0.4846
     20 components      0.5489
     30 components      0.5531
     40 components      0.4642
```

**PCA improved the clustering: 0.468 with all 64 columns against 0.553 with 30
components** — about an 18% relative gain, from throwing away half the columns.

This is the opposite of Question 5, where PCA only cost accuracy. The difference is the
method, and it is worth understanding rather than memorising:

**K-Means measures distance, and distance in 64 dimensions is noisy.** Every one of those
64 pixel columns contributes to the gap between two images, including the corner pixels
that are blank in almost every digit. Those columns add variation that is not information,
and because distance *sums* over all columns, that noise accumulates. Logistic regression
in Question 5 could just assign a useless column a small weight — **K-Means has no weights
to assign.** Every column counts equally, whether it deserves to or not.

**Now look at the shape of the column: it rises, peaks, and falls.**

- **2 components is far too few** (0.326). Question 9 showed you why — 8, 9 and 5 are piled
  on top of each other in that view.
- **30 is the best of those tried** (0.553).
- **40 drops back to 0.464**, essentially the no-PCA score. The noise is coming back in.

So there is a genuine sweet spot, it is nowhere near either end, and **you only find it by
trying.** That is Question 6's lesson arriving again through a different door.

**The rule this supports:** PCA tends to help distance-based methods — K-Means, kNN,
DBSCAN — on wide data, and tends not to help models that already fit a weight per column.
That is a tendency, not a law. Both of the last two questions were measured, not assumed.

## Question 12

**Argue against PCA.**

A colleague proposes running PCA on every dataset the team works with, as a standard first
step, "because it removes noise and speeds everything up".

Write **four or five sentences** against that policy, using evidence from this notebook.
Then say the one situation in which you would agree with them.

### Answer

**An argument that holds up:**

> *"A blanket PCA step would cost us more than it saves. On the breast cancer data PCA
> never beat the raw columns — 0.9807 against 0.9789 at ten components — so on a 30-column
> dataset we would be paying accuracy for a speed problem we do not have. It also destroys
> our ability to explain a model: we can tell a clinician that 'mean concave points' drives
> a prediction, but not '0.26 × concave points + 0.26 × concavity + 28 more terms'. And it
> is not safe by default — in Question 10 the component holding 0.27% of the variance was
> the only one that predicted the label, so a standard 95% rule would have silently cut
> accuracy from 0.988 to 0.552. PCA also assumes the columns are numeric and comparable,
> which breaks the moment we have one-hot categories in the frame."*
>
> *"Where I would agree with them: wide data going into a distance-based model. On the
> digits, K-Means went from 0.468 to 0.553 with thirty components, and kNN on 300 columns
> got about five times faster at identical accuracy. For those cases PCA should be a
> candidate — but as a step we tune inside cross-validation, not a reflex we apply before
> looking."*

The evidence, laid out:

| Claim in the proposal | What this notebook showed |
|---|---|
| "removes noise" | Sometimes. It removed noise for K-Means (0.468 → 0.553) and removed the *signal* in Question 10 (0.988 → 0.542). It cannot tell the difference, because it never sees the label. |
| "speeds everything up" | Only where columns are the bottleneck. 30 columns on 569 rows has no speed problem to solve, and PCA costs a fit. |
| "as a standard first step" | This is the real error. Applied before looking, it cannot be judged — and it is lossy and irreversible in the pipeline. |

Two further costs the proposal does not account for:

- **Interpretability**, which on medical or financial data is often a hard requirement
  rather than a nicety. A component is a weighted blend of every column; "why did the model
  say that" has no short answer any more.
- **Leakage risk.** Done casually, PCA gets fitted on the whole dataset before the split.
  The components then carry information from the test rows, and the scores come back
  flattering and wrong. Inside a `Pipeline` this is handled; as an informal "first step" it
  frequently is not.

**The underlying principle, which is bigger than PCA:** a technique applied by default is a
technique nobody is evaluating. The right form of the proposal is *"PCA should be in our
standard set of things to try, tuned inside cross-validation"* — which is a different
policy, and a defensible one.

---

## Submission

Run **Cell → Run All** on a fresh kernel and confirm there are no errors, then upload the
notebook. Questions 3, 6, 9 and 12 are marked mostly on the explanation — a correct number
with no reasoning gets little credit.